# core

> core provides dialog introspection, message linking, tool discovery, and import/export utilities for solveit dialogs.

In [ ]:
#| default_exp dutil

In [ ]:
gk = tuple((g := globals()).keys())

In [ ]:
#| export
#| hide
import os, re, sys, inspect, uuid, time
from collections import defaultdict
from inspect import Parameter, currentframe
from pathlib import Path
from typing import Any, Mapping
from IPython import get_ipython
import tiktoken
from anyio import sleep
from anyio.from_thread import start_blocking_portal
from fastcore.imports import in_ipython
from fastcore.meta import delegates
from fastcore.xtras import is_listy
import dialoghelper
from dialoghelper.core import add_msg, is_usable_tool, msg_idx, read_msg, update_msg, find_msgs, find_dname
from toolslm.funccall import get_schema, resolve_nm
from fastgit import Git

In [ ]:
#| hide
from tempfile import TemporaryDirectory
from IPython.display import Markdown
from fastcore.test import *
from fastcore.xtras import dict2obj
import dialoghelper.tmux

## globals - what's available on startup?
Exploring solveit sym injections

Run below first after a restart.

In [ ]:
print(gk)

('__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', 'read_gh_repo', 'read_url', 'transient', 'run_cmd', 'maybe_await', 'call_tool', 'sys', 'greet', 'craft_chain', '_i', '_ii', '_iii', '_i1', '_1', '_i2', '__os', '_i3', '_i4', '__dialog_name', '_i5', 'pyrun', 'allow', 'core', 'dname_doc', 'md_cls_d', 'dh_settings', 'Placements', 'mermaid_url', 'msg_insert_line', 'msg_str_replace', 'msg_strs_replace', 'msg_replace_lines', 'msg_del_lines', 'msg_pyrun', 'msg_ast_replace', 'add_styles', 'find_dname', 'xposta', 'xgeta', 'call_endp', 'call_endpa', 'curr_dialog', 'msg_idx', 'add_html_a', 'add_html', 'add_scr_a', 'add_scr', 'iife_a', 'iife', 'add_mod', 'add_mod_a', 'pop_data_a', 'pop_data', 'fire_event_a', 'fire_event', 'event_get_a', 'event_get', 'trigger_now', 'event_once', 'event_once_a', 'js_run', 'js_run_a', 'js_eval', 'js_eval_a', 'display_response', 'conn

In [ ]:
if 'find_dname' in gk: assert f"/{__dialog_name}" == find_dname()
print(f"{g['__name__']=}\n{g['transient']=}\n{g['__os']=}\n{g['__dialog_name']=}")

g['__name__']='__main__'
g['transient']=<function transient at 0x7b5fb0d43920>
g['__os']=<module 'os' from '/usr/local/lib/python3.12/os.py'>
g['__dialog_name']='prj/pote/nbs/00_dutil'


In [ ]:
# dir(ipykernel_helper)

In [ ]:
# await add_msg(symsrc('ipykernel_helper.core'), msg_type='raw')

In [ ]:
sys.meta_path

 _frozen_importlib.BuiltinImporter,
 _frozen_importlib.FrozenImporter,
 _frozen_importlib_external.PathFinder,
 __editable___droute_0_0_1_finder._EditableFinder,
 __editable___htmx_bridge_0_0_1_finder._EditableFinder,
 __editable___pote_0_1_1_finder._EditableFinder,
 __editable___solveit_dmtools_0_0_29_finder._EditableFinder,
 __editable___solveit_0_0_105_finder._EditableFinder,

In [ ]:
[k for k in sys.modules if 'solveit' in k]

['__editable___solveit_dmtools_0_0_29_finder',
 '__editable___solveit_0_0_105_finder']

In [ ]:
slvt_k = [k for k in sys.modules if '__editable___solveit_' in k][0]
print(dir(slvt := sys.modules[slvt_k]))

['MAPPING', 'ModuleSpec', 'NAMESPACES', 'PATH_PLACEHOLDER', 'Path', 'PathFinder', '_EditableFinder', '_EditableNamespaceFinder', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'chain', 'install', 'module_suffixes', 'spec_from_file_location', 'sys']


In [ ]:
# if (msg := await read_msg(1, True))['content'].startswith('!cat'):
#     update_msg(msg['id'], content=f"!cat {slvt.__file__}")
#     run_msg(msg['id'])

In [ ]:
' '.join(_.__module__ for _ in sys.meta_path)

'_distutils_hack _frozen_importlib _frozen_importlib _frozen_importlib_external __editable___droute_0_0_1_finder __editable___htmx_bridge_0_0_1_finder __editable___pote_0_1_1_finder __editable___solveit_dmtools_0_0_29_finder __editable___solveit_0_0_105_finder six importlib_metadata'

In [ ]:
# s = Path('/usr/local/lib/python3.12/site-packages/__editable___solveit_0_0_89_finder.py').read_text()
# await add_msg(s, msg_type='raw')

In [ ]:
#| export
def solveit_version():
    "Return the version of solveit if it is found"
    s = ' '.join(_.__module__ for _ in sys.meta_path)
    mtch = re.match(r'.*__editable___solveit_(\d+)_(\d+)_(\d+)_finder', s)
    return f"{mtch[1]}.{mtch[2]}.{mtch[3]}" if mtch else ''

In [ ]:
solveit_version()

'0.0.105'

In [ ]:
os.environ.get('IN_SOLVEIT')

'True'

In [ ]:
#| export
def in_dialog():
    "Check if the code is running in a solveit dialog"
    if not (os.environ.get('IN_SOLVEIT') and in_ipython() and bool(solveit_version())): return False
    try:
        if find_dname(): return True
    except: pass
    return False

In [ ]:
in_dialog()

True

In [ ]:
#| export
def get_caller_globals(): 
    "Return the globals of the caller"
    return inspect.currentframe().f_back.f_globals

In [ ]:
test_var = "I'm in globals"
def caller_func():
    local_var = "I'm local"
    return get_caller_globals()

test_is('test_var' in caller_func(), True)
test_is('local_var' in caller_func(), False)

## helpers

### context usage
Measures how many tokens from the current dialog have been consumed by the LLM up to a given message position.

In [ ]:
#| export
def count_tokens(src, model='cl100k_base'):
    "Count tokens in `src` using `model` encoding"
    return len(tiktoken.get_encoding(model).encode(src))

Use gpt4 tokenizer by default.

In [ ]:
count_tokens("def foo(x:int) -> bool: return x > 0")

13

In [ ]:
#| export
async def ctxusage(id:str='', dname:str=''):
    msgs = await find_msgs(include_output=False, dname=dname)
    id = id or (await read_msg(0)).id
    pos = await msg_idx(id)
    return sum(m.input_tokens + (m.output_tokens or 0) for m in msgs[:pos] if not m.skipped)

In [ ]:
await ctxusage()

4945

### find/set_var

In [ ]:
#| export
# old version from `dialoghelper`, not thread safe
def _find_frame_dict(sentinel:str):
    "Find the globals dict containing sentinel, or calling frame's globals if no sentinel"
    frame = currentframe().f_back.f_back
    if not sentinel: return frame.f_globals
    while frame:
        if sentinel in frame.f_globals: return frame.f_globals
        frame = frame.f_back
    return globals()

def set_var(var:str, val, force:bool=False):
    "Set var to val after finding it in all frames of the call stack"
    frame = _find_frame_dict(var)
    if var not in frame and not force: raise ValueError(f"Could not find {var} in any scope")
    frame[var] = val

def find_var(var:str, default:Any=Parameter.empty):
    "Search for var in all frames of the call stack"
    if var in (frame := _find_frame_dict(var)): return frame[var]
    return default
find_var.notfound = Parameter.empty

In [ ]:
test_is(find_var('aSdF'), inspect._empty)
test_is(find_var('aSdF', None), None)

set_var('aSdF', 13, True)
test_is(aSdF,13)
with ExceptionExpected(): set_var('aSdFgHj', 17)

### message linking

see <a href="/dialog_?name=prj%2Fdutil%2Fexplorer%2Flinked_msg" target="_blank">linked_msg explorer</a>

In [ ]:
#| export
_ts, _te = ('<!-- ', '#| '), (' -->', '')
async def get_tag(name:str, val:str='', msg_type:str='note'):
    return f"{_ts[msg_type=='code']}{name}: {val or (await read_msg(0)).id}{_te[msg_type=='code']}"

In [ ]:
#| export
_tagpats = {
    'code': (re.compile(r'\A#\|\W*(\w+): ([_a-f0-9]{9})\W*$', re.MULTILINE),
             re.compile(r'^#\|\W*(\w+): ([_a-f0-9]{9})\W*\Z', re.MULTILINE)),
    'note': (re.compile(r'\A<!-- (\w+): ([_a-f0-9]{9}) -->\W*$', re.MULTILINE),
             re.compile(r'^<!-- (\w+): ([_a-f0-9]{9}) -->\s*\Z', re.MULTILINE))
}
_tagpats['raw'] = _tagpats['prompt'] = _tagpats['note']
_tagpats[None] = _tagpats['note'] + _tagpats['code']

def has_tag(s:str, msg_type:str=None) -> bool:
    "Check if string contains tags created by get_tag at the start or end of the string"
    if msg_type == 'code' and not (s.startswith('#|') or '\n#|' in s): return False
    if msg_type == 'note' and not (s.startswith('<!--') or '\n<!--' in s): return False
    return any(re.search(pat, s) for pat in _tagpats[msg_type])

In [ ]:
#| export
def find_tag(s: str, msg_type: str = None) -> str:
    "Find tag in a string and return their details"
    for pat in _tagpats[msg_type]:
        if match := re.search(pat, s): return f"{match.group(1)}: {match.group(2)}"
    return ''

In [ ]:
#| export
async def get_linked(id:str) -> str:
    if (msg := (await read_msg(0, id=id))).get('id','') == id:
        if tag := find_tag(msg.content, msg.msg_type):
            k,v = tag.split(': ')
            if k == 'linkedto': return v
    return ''

In [ ]:
#| export
if in_dialog() and __name__ != '__main__': get_ipython().xpush(__linked_msgs={})

In [ ]:
#| export
delegates(add_msg)
async def link_msg(
    content:str=None,  # Content of the linked message
    id:str=None,  # ID of the message to link to, or current message if not provided
    pos:str='end',  # Position of the tag in the message
    **kwargs  # Additional keyword arguments for `add_msg` or `update_msg`
) -> str:  # id of linked message
    "Add or update a message linked to `anchor` message. Note only one linked msg per anchor."
    anchor_id, linked = id or (await read_msg(0)).id, find_var('__linked_msgs')
    def _with_tag(tag, c): return f"{tag}\n{c}" if pos=='start' else f"{c}\n{tag}"
    if linked_id := linked.get(anchor_id):
        if (msg := await read_msg(0, id=linked_id)).get('id','') == linked_id:
            tag = await get_tag('linkedto', anchor_id, kwargs.get('msg_type', msg.msg_type))
            if content: kwargs['content'] = _with_tag(tag, content)
            linked[anchor_id] = await update_msg(linked_id, **kwargs)
            return linked[anchor_id]
    tag = await get_tag('linkedto', anchor_id, kwargs.get('msg_type', 'note'))
    linked[anchor_id] = await add_msg(_with_tag(tag, content or '.'), id=anchor_id, **kwargs)  # NOTE: bug note message w/ only comment
    return linked[anchor_id]

In [ ]:
linkedid = await link_msg("I'm linked to my destiny!")

I'm linked to my destiny!
<!-- linkedto: _3ba03e96 -->

In [ ]:
thisidx = await msg_idx()
await run_msg(linkeeid := (await read_msg(-2)).id)
test_eq(thisidx, await msg_idx(linkeeid)+2)

In [ ]:
#| export
async def hydrate():
    "Traverse dialog looking for linked messages to update `__linked_msgs`"
    linked = find_var('__linked_msgs')
    linked.clear()
    msgs = await find_msgs(include_meta=True, include_output=False, include_skipped=True)
    ids = {_.id for _ in msgs}
    for msg in msgs:
        if tag := find_tag(msg.get('content', ''), msg.get('msg_type')):
            k, v = tag.split(': ')
            if k == 'linkedto' and v in ids: linked[v] = msg['id']
    return linked

In [ ]:
links = await hydrate()
test_eq(links[linkeeid], linkedid)
display(links)

{'_3ba03e96': '_af955df6',
 '_89419261': '_c2797f3b',
 '_f40fb050': '_564053c7',
 '_9bd38272': '_b41df4e6',
 '_11232531': '_92c41434',
 '_c877d4ad': '_2c5bc08f'}

### wait
> helpers for blocking call w/out freezing the kernel

In [ ]:
# def nq(cmd, *args, **kwargs):
#     """Queue the run of `cmd(*args, **kwargs)` and wait until dialog state settles.
#     
#     Ensures that after each call, the dialog state on the Solveit server is settled
#     before returning. For `run_msg`, polls asynchronously to avoid blocking kernel execution.
#     
#     Note: While this ensures server-side state is settled, the frontend may lag slightly
#     in displaying changes. If messages don't appear immediately, refresh the page or click
#     the green websocket status button. Often, a small time.sleep after this call will alleviate the issue.
#     """
#     res = cmd(*args, **kwargs)
#     if cmd.__name__ == 'run_msg':
#         ids = kwargs.get('ids') if 'ids' in kwargs else (args[0] if args else None)
#         if not ids: ids = find_msg_id()
#         async def poll_exe():
#             while True:
#                 msgs = find_msgs(ids=ids, include_meta=True, include_output=False)
#                 if all(m.get('time_run') for m in msgs): return
#                 await sleep(0.05)
#         with start_blocking_portal() as portal:  portal.call(poll_exe)
#         return res
#     if cmd.__name__ in ('add_msg', 'update_msg', 'del_msg'): return res  # Already settled by the time the call returns
#     return res  # Unknown command, just return

In [ ]:
#| export
def waitpred(pred, timeout=0.5, interval=0.05):
    "Wait until `pred` is True or `timeout` is reached without blocking the kernel"
    async def _wait():
        start = time.time()
        while time.time()-start < timeout and not pred(): await sleep(interval)
        return pred()
    with start_blocking_portal() as portal: return portal.call(_wait)

In [ ]:
#| export
async def waitpreda(pred, timeout=0.5, interval=0.05):
    "Async wait until `pred` is True or `timeout` is reached without blocking the kernel"
    t0 = time.time()
    while not await pred():
        if time.time() - t0 > timeout: return False
        await sleep(0.1)
    return True

In [ ]:
# %%time
# test_eq(waitpred(lambda: True), True)

In [ ]:
# %%time
# test_eq(waitpred(lambda: False, timeout=0.3), False)

In [ ]:
# %%time
# x = 0
# def delayed(): 
#     global x; x += 1; return x > 2
# test_eq(waitpred(delayed, timeout=0.5), True)
# test_is(x>2, True)

In [ ]:
# y = []
# def sideeffect(): y.append(1); return len(y) > 5
# waitpred(sideeffect, timeout=0.3)
# test_is(len(y) > 2, True)

In [ ]:
# y = []

In [ ]:
# if y:
#     for msgid in y: await del_msg(msgid)
#     y.clear()
# 
# async def sideeffect(): y.append(await add_msg(str(y))); return len(y) > 2
# await waitpreda(sideeffect, timeout=4)
# test_is(len(y) > 2, True)

In [ ]:
# for _ in reversed(y): await del_msg(_)

In [ ]:
# test_fail(lambda: waitpred(lambda: 1/0, timeout=0.2), False)

In [ ]:
# y = []

### next dupe in folder

In [ ]:
#| export
def next_dup(fp, marker='_dup'):
    "Get next available duplicate number in fp parent directory"
    fp = Path(fp)
    p, nm, suff = fp.parent, fp.stem.split('.')[0], ''.join(fp.suffixes)
    nm = m[1] if (m := re.match(r'^(.+)'+re.escape(marker)+r'\d+$', nm)) else nm
    # if not p.exists(): i, p/f"{nm}{marker}{i}{suff}"
    dupr = re.compile(r'^'+re.escape(nm+marker)+r'(\d+)$')
    nums = sorted({int(m[1]) for f in p.iterdir() if (m := re.match(dupr, f.stem.split('.')[0]))})
    # if not nums: return 0, p/f"{nm}{suff}"
    for i,n in enumerate(nums, 1):
        if i != n: return i, p/f"{nm}{marker}{i}{suff}"
    return len(nums)+1, p/f"{nm}{marker}{len(nums)+1}{suff}"



A given directory can have files with names that match "*_dup1.*", "*_dup2.*", etc. There can be gaps in the sequence, e.g., 1, 2, 5, 7. `next_dup` get the next available number starting from 1 and considering gaps.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img.jpg', 'img_dup3.jpg'): (p/_).touch()
    n, fp = next_dup(p/'img.jpg')
test_eq((n, fp), (1, p/'img_dup1.jpg'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img.jpg', 'img_dup1.jpg'): (p/_).touch()
    n, fp = next_dup(p/'img.jpg')
test_eq((n, fp), (2, p/'img_dup2.jpg'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('test_dup1.txt', 'test_dup2.txt', 'test_dup5.txt', 'test_dup7.txt'): (p/_).touch()
    n, fp = next_dup(p/'test_dup2.txt')
test_eq((n, fp), (3, p/'test_dup3.txt'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    (p/'_test_dup23.txt').touch()
    n, fp = next_dup(p/'test_dup3.txt')
test_eq((n, fp), (1, p/'test_dup1.txt'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img_dup2.jpg', 'img_dup3.jpg'): (p/_).touch()
    n, fp = next_dup(p/'test_dup3.txt')
test_eq((n, fp), (1, p/'test_dup1.txt'))

In [ ]:
#| export
def next_filename(path:str):
    "Get next available duplicate filename for path, e.g., 'img.jpg' -> 'img_dup1.jpg'"
    _,fp = next_dup(Path(path))
    return str(fp)

### simple ids

In [ ]:
#| export
def gen_id(): return f"_{uuid.uuid4().hex[:8]}"

In [ ]:
gen_id()

'_43c2861b'

### object traversal

In [ ]:
#| export
_empty = inspect.Parameter.empty

In [ ]:
def val_at(o, sym: str, default=_empty, sep='.'):
    "Traverse nested `o` looking for attributes/items specified in dot-separated `sym`."
    if not isinstance(sym, str): raise TypeError(f'{sym=!r} is not a string')
    try:
        for a in sym.split(sep):
            if a[0]=='-' or a[0].isdigit(): a = int(a)
            try: o = o[a]
            except Exception:
                if isinstance(a, int):
                    a = str(a)
                    try: o = o[a]
                    except Exception: pass
                o = getattr(o, a)
    except Exception as e:
        if default is not _empty: return default
        raise e
    return o

In [ ]:
records = [
    {'id': 1, 'items': [2, 3], 'meta': {'count': 4}}, 
    {'id': 5, 'items': [6, 7], 'nested': [{'val': 81}, {'val': 82}]}, 
    {'id': 9, 'items': [10, 11], 'meta': {'count': 12}}
]

test_eq(val_at(records, '0.id'), 1)
test_eq(val_at(records, '1.items'), [6, 7])
test_eq(val_at(records, '2.meta'), {'count': 12})

Works with lists of dicts (common in API responses)


In [ ]:
test_fail(lambda: val_at({}, 'a.b'))
test_fail(lambda: val_at([], 'a.b'))
test_fail(lambda: val_at({'a': 1}, 'a.b'))
test_fail(lambda: val_atpath({'a': 1}, 'a', 'b'))

# With default, no error
test_eq(val_at({'a': 1}, 'a.b', None), None)

Error handling: raises when path not found (unless default provided)

In [ ]:
j2 = {
    "app": {
        "Garden": {
            "Flowers": {
                "Red flower": "Rose",
                "White Flower": "Jasmine",
                "Yellow Flower": "Marigold"
            }
        },
        "Fruits": {
            "Yellow fruit": ["Mango", {"Banana": ["Canary Island", "Puerto Rico"]}],
            "Green fruit": "Guava",
            "White Flower": "groovy"
        },
        "Trees": {
            "label": {
                "Yellow fruit": "Pumpkin",
                "White Flower": "Bogan"
            }
        },
        "Numbers": [1, 2, 3, 4, 5],
        "Boolean": True,
        "Null": None
    }
}

j2_str = j2#json.dumps(j2)

test_eq(val_at(j2_str, 'app.Fruits.Yellow fruit.1.Banana.0'), 'Canary Island')
test_eq(val_at(j2_str, 'app.Garden.Flowers.Red flower'), 'Rose')
test_eq(val_at(j2_str, 'app.Numbers.2'), 3)
test_eq(val_at(j2_str, 'app.Boolean'), True)
test_eq(val_at(j2_str, 'app.Null'), None)
test_fail(lambda: val_at(j2_str, 'app.NonExistent'))
test_fail(lambda: val_at(j2_str, 'app.Fruits.Yellow fruit.3'))
test_is(val_at(j2_str, 'app.Fruits.Yellow fruit.3', None), None)

In [ ]:
j2_obj = dict2obj(j2)

test_eq(val_at(j2_obj, 'app.Fruits.Yellow fruit.1.Banana.0'), 'Canary Island')

New version of `val_at` that also acepts indexing, i.e. sym can be dot separated with indexing:
- app.Fruits.Yellow fruit[1].Banana[0] <=> app.Fruits.Yellow fruit.1.Banana.0
- app[Fruits][Yellow fruit][1][Banana][0] <=> app.Fruits.Yellow fruit.1.Banana.0
- app[Fruits].Yellow fruit.1.Banana[0] <=> app.Fruits.Yellow fruit.1.Banana.0
- 2.meta <=> [2].meta
- meta.2 <=> meta[2]

In [ ]:
#| export
def at_(
    o, # Object to traverse (dict, list, object, or nested combination)
    sym: str, # Path using dots and/or brackets (e.g., 'a.b[0].c' or 'a[b][c]')",
    default: Any=_empty, # Value to return if path not found (raises exception if not provided)
    sep='.' # Separator for path segments
) -> Any: # Value at the specified path
    "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    try:
        for a in filter(None, sym.split(sep)):
            if a.lstrip('-').isdigit(): a = int(a)
            try: o = o[a]
            except Exception:
                if isinstance(a, int):
                    try: o = o[str(a)]; continue
                    except Exception: pass
                o = getattr(o, a)
    except Exception:
        if default is not _empty: return default
        raise
    return o

`at_` provides flexible path-based access to nested data structures:

**Supported types:** `o` can be/contains any combination of `Sequence`, `Mapping` (dicts, lists, tuples, L, etc), objects with \_\_getitem__, and/or dataclasses, objects with attributes accesible by `getattr`.

**Path syntax:**
- Dot notation: `'a.b.c'` accesses `o['a']['b']['c']` or `o.a.b.c`
- Bracket notation: `'a[b][c]'` is equivalent to `'a.b.c'`: `[x]` is a shorthand for `.x`
- Mixed: `'a.b[0].c'` combines both styles
- Numeric indices: `'items.2'` or `'items[2]'` for list/array access
- Empty path: `''` returns the object itself

**Access priority:** Item access (`[]`) is tried before attribute access (`.`)

**Error handling:** Raises exception if path not found, unless `default` is provided

In [ ]:
test_eq(at_({'a': 13}, 'a'), 13)

test_eq(at_({'a': {'b': 13}}, 'a.b'), 13)
test_eq(at_({'a': dict2obj({'b': 13})}, 'a.b'), 13)

test_eq(at_({'a': {'3': 7}}, 'a.3'), 7)

test_fail(lambda: at_(o, 'app.NonExistent'))
test_fail(lambda: at_(o, '[0'))
test_fail(lambda: at_(None, 'a'))
test_eq(at_(None, 'a', None), None)
test_fail(lambda: at_(None, 'a'))

test_eq(at_(o := {'meta': [1,2,3]}, ''), o)
test_eq(at_(o, 'meta[2]'), 3)
test_eq(at_(o, 'meta.2'), 3)

test_eq(at_('xyz', 'a', None), None)
test_eq(at_((s := 'xyz'), 'split'), s.split)

test_eq(at_([{'a':1}], '[0][a]'), 1)

test_eq(at_(records, '[2].meta'), {'count': 12})
test_eq(at_(records, '2[meta]'), {'count': 12})

j2_list = [{'a':1}, {'b':2}]
test_eq(at_(j2_list, '0'), {'a':1})
test_eq(at_(j2_list, '1'), {'b':2})
test_eq(at_(j2_list, '[0][a]'), 1)
test_eq(at_(j2_list, '0.a'), 1)
test_eq(at_(j2_list, '[1].b'), 2)

test_eq(at_(j2, 'app[Numbers][2]'), 3)
test_eq(at_(j2, 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(j2, 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_eq(at_(dict2obj(j2), 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(dict2obj(j2), 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_is(at_(j2, '[bad][path]', 'default'), 'default')
test_is(at_(j2, '[missing]', None), None)
test_is(at_(j2, 'app[missing]', 'default'), 'default')
test_is(at_(j2, 'app[NonExistent]', None), None)

In [ ]:
print(13)

13


In [ ]:
test_eq(at_(await read_msg(), 'output').strip(), '13')

I'm a note.

In [ ]:
test_eq(at_(await read_msg(), 'output'), '')

In [ ]:
if not at_(await read_msg(1), 'content').startswith("I'm a prompt"):
    msgid = await add_msg("I'm a prompt", msg_type='prompt', output='are you? really?')
    # _=add_scr(f"selectMsg($('#{msgid}'), {centered: true})")
display(Markdown('### Test markdown'))
print(_msgid := (await read_msg(0)).id)

<div class="prose">

### Test markdown

</div>

_86e4f755


I'm a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

are you? really?

In [ ]:
test_eq(at_(await read_msg(), 'output'), 'are you? really?')

I'm a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

are you? really?

In [ ]:
oo = at_(await read_msg(0, True, _msgid), 'output')
oo

'<div class="prose">\n\n### Test markdown\n\n</div>\n_86e4f755\n'

Future improvements, more complex paths:

1. **Negative indexing support** - currently `-1` works but only because Python handles it; we could make it explicit
2. **Wildcard/glob patterns** - e.g. `items.*.name` to get all names from a list of dicts
3. **Optional chaining** - e.g. `a?.b?.c` returns `None` instead of raising if any part is missing
4. **Slice support** - e.g. `items[1:3]` or `items[:2]`
5. **Multiple paths** - return values from multiple paths at once
6. **Callable filter** - e.g. `items[lambda x: x['active']].name`

Can we improve `_at` somehow? Give me alternatives. I'm interested in conciseness, strength, flexibility; more complex paths will be handle by another function.

The main opportunity is extracting a `_step` helper to flatten the nested try/except. Here are two alternatives:

**Option A** — helper + loop (minimal change, much cleaner):

```python
def _step(o, k):
    try: return o[k]
    except Exception:
        if isinstance(k, int):
            try: return o[str(k)]
            except Exception: pass
        return getattr(o, k)

def at_(o, sym, default=_empty):
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    try:
        for a in filter(None, sym.split('.')): o = _step(o, int(a) if a.lstrip('-').isdigit() else a)
    except Exception:
        if default is not _empty: return default
        raise
    return o
```

**Option B** — functional style with `reduce`:

```python
def at_(o, sym, default=_empty):
    ks = [int(a) if a.lstrip('-').isdigit() else a for a in filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split('.'))]
    try: return reduce(lambda o,k: _step(o,k), ks, o)
    except Exception:
        if default is not _empty: return default
        raise
```

Both drop the `sep` param since complex paths are handled elsewhere. Option A is more readable; Option B is more functional/compact but the list comprehension on one line gets long. I'd lean toward **Option A** — the `_step` helper pays for itself in clarity.

**1. Reduce-based (most concise)**

In [ ]:
# def at_(o, sym, default=_empty, sep='.'):
#     "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
#     def step(o, a):
#         if a.lstrip('-').isdigit(): a = int(a)
#         try: return o[a]
#         except: return getattr(o, str(a))
#     try: return reduce(step, filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split(sep)), o)
#     except: 
#         if default is not _empty: return default
#         # raise

**2. Operator.getitem + getattr combo (cleaner step logic)**

In [ ]:
# def at_(o, sym, default=_empty, sep='.'):
#     "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
#     def step(o, a):
#         if a.lstrip('-').isdigit(): a = int(a)
#         for fn in (lambda: o[a], lambda: o[str(a)] if isinstance(a,int) else None, lambda: getattr(o,a)):
#             try: return fn()
#             except: pass
#         raise KeyError(a)
#     try: return reduce(step, filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split(sep)), o)
#     except:
#         if default is not _empty: return default
#         raise

### add_to_namespace

In [ ]:
#| export
async def setup_ns(ns=None, **kwargs):
    "Add `kwargs` to the namespace `ns` or current dialog"
    ns = ns or _find_frame_dict('__dialog_name')
    # lc = list(locals().items())[1:]
    thisid = (await read_msg(0)).id
    for k,v in kwargs.items(): ns[k] = v
    msgid = await link_msg('\n\n'.join(f"{k} = {v}" for k,v in kwargs.items()))
    print(thisid, msgid)

setup_dialog = setup_ns

In [ ]:
await setup_ns(
    ORG='', 
    REPO='', 
    USERNAME='civvic', 
    DESCRIPTION='', 
    COMMIT_MESSAGE='', 
    BRANCH_NAME='',
)

_89419261 _c2797f3b


ORG = 

REPO = 

USERNAME = civvic

DESCRIPTION = 

COMMIT_MESSAGE = 

BRANCH_NAME = 
<!-- linkedto: _89419261 -->

In [ ]:
test_is('USERNAME' in globals(), True)

## info

In [ ]:
g = Git('.')

In [ ]:
g('status')

['On branch dev-dutil',
 "Your branch is ahead of 'origin/dev-dutil' by 1 commit.",
 '  (use "git push" to publish your local commits)',
 '',
 'Changes not staged for commit:',
 '  (use "git add <file>..." to update what will be committed)',
 '  (use "git restore <file>..." to discard changes in working directory)',
 '\tmodified:   ../explorer/dialoghelper_patcher.ipynb',
 '\tmodified:   00_basic.ipynb',
 '\tmodified:   00_dutil.ipynb',
 '\tmodified:   01_dialog.ipynb',
 '\tmodified:   02_git.ipynb',
 '',
 'Untracked files:',
 '  (use "git add <file>..." to include in what will be committed)',
 '\t../explorer/agent-as-infraestructure-builder.ipynb',
 '',
 'no changes added to commit (use "git add" and/or "git commit -a")']

In [ ]:
g('status', '-bs')

['## dev-dutil...origin/dev-dutil [ahead 1]',
 ' M ../explorer/dialoghelper_patcher.ipynb',
 ' M 00_basic.ipynb',
 ' M 00_dutil.ipynb',
 ' M 01_dialog.ipynb',
 ' M 02_git.ipynb',
 '?? ../explorer/agent-as-infraestructure-builder.ipynb']

In [ ]:
#| export
def info(dname:str='', json:bool=False):
    "Returns information about the dialog"
    br, chngs, ver, dhv = (), (), solveit_version(), dialoghelper.__version__
    g = Git('.' if not dname else (Path.home()/find_dname(dname).lstrip('/')).parent) 
    if g.exists:
        br, *chngs = g('status', '-bs')
        br = br.split()[1].split('...')[0]
    return ({'Solveit': solveit_version(), 'dialoghelper': dialoghelper.__version__, 'git branch': br, 'git changes': chngs}
        if json else 
        f"Solveit: **v. {ver}**  \ndialoghelper: **v. {dhv}**  " + f"\ngit branch: **{br}**  \ngit changes: {chngs}" if br else '')

In [ ]:
Markdown(info('/prj/EXPER/expers/curator/curator'))

<div class="prose">

Solveit: **v. 0.0.105**  
dialoghelper: **v. 0.2.31**  
git branch: **main**  
git changes: [' M ../../.gitignore', '?? ../../CRAFT.ipynb', '?? ../../__init__.py', '?? ../../meta-EXPER-v001.ipynb', '?? ../../meta-EXPER.ipynb', '?? ../../meta-EXPER_DEV.ipynb', '?? "../../what is an EXPER.ipynb"']

</div>

In [ ]:
Markdown(info())

<div class="prose">

Solveit: **v. 0.0.105**  
dialoghelper: **v. 0.2.31**  
git branch: **dev-dutil**  
git changes: [' M ../explorer/dialoghelper_patcher.ipynb', ' M 00_basic.ipynb', ' M 00_dutil.ipynb', ' M 01_dialog.ipynb', ' M 02_git.ipynb', '?? ../explorer/agent-as-infraestructure-builder.ipynb']

</div>

In [ ]:
#| export
async def add_info(msgid:str='', dname:str=''):
    "Add a message with information about the dialog"
    return await link_msg(info(dname), id=msgid)

In [ ]:
await add_info();

Solveit: **v. 0.0.105**  
dialoghelper: **v. 0.2.31**  
git branch: **dev-dutil...origin/dev-dutil**  
git changes: [' M ../explorer/dialoghelper_patcher.ipynb', ' M 00_basic.ipynb', ' M 00_dutil.ipynb', ' M 01_dialog.ipynb', ' M 02_git.ipynb', '?? ../explorer/agent-as-infraestructure-builder.ipynb']
<!-- linkedto: _f40fb050 -->

## summarize

In [ ]:
#| export
def summarize(target, context): pass

## tools

### tool info

In [ ]:
# fc_tool_info()

In [ ]:
# tool_info()

In [ ]:
# inspect_tool_info()

In [ ]:
# dialoghelper.tmux.tmux_tool_info()

In [ ]:
def get_tool_names(ns): return [k for k,v in ns.items() if callable(v)]

In [ ]:
# print(list(get_tool_names(globals())))

see <a href="/dialog_?name=prj%2Fdutil%2Fexplorer%2F00_core_isolated" target="_blank">get_tool_names isolated</a>

In [ ]:
#| export
def _get_ns(ns):
    if inspect.ismodule(ns): ns = vars(ns)
    elif isinstance(ns, str): ns = {k: get_ipython().user_ns[k] for _ in re.split(r'\s*,\s*', ns) if (k := _.strip())}
    return ns or get_ipython().user_ns

def get_tool_names(
    ns:Mapping|str=None,  # module,mapping,comma-separated str; None uses IPython user namespace
    exclude:Mapping|list[str]=None,  # module/mapping (recursively scanned) or list of symbol names to exclude
    only_exported:bool=False,  # if ns is a module, only include symbols in __all__
    exclude_private:bool=True  # exclude symbols starting with '_'
) -> dict[str,list[str]]:  # module name -> list of tool names
    "Return dict mapping module names to lists of usable tool names from namespace ns (or IPython user namespace if None)."
    ns = _get_ns(ns)
    if exclude: exclude = set(sum(get_tool_names(exclude).values(), []) if not is_listy(exclude) else exclude)
    res, vis, exports = defaultdict(list), defaultdict(set), set(ns.get('__all__', []))
    for k,v in ns.items():
        if exclude_private and k[0] == '_': continue
        if only_exported and k not in exports: continue
        if exclude and k in exclude: continue
        if not hasattr(__builtins__, k) and callable(v):
            try:
                if is_usable_tool(v): 
                    if inspect.isclass(v) and '__call__' not in v.__dict__: continue
                    if get_schema(v): 
                        mod = getattr(v, '__module__', 'unknown')
                        if v not in vis[mod]: res[mod].append(k); vis[mod].add(v)
            except Exception: pass
    return dict(res)

In [ ]:
get_tool_names()

{'ipykernel_helper.core': ['read_gh_repo', 'read_url'],
 'fastcore.tools': ['run_cmd',
  'ensure',
  'valid_path',
  'rg',
  'sed',
  'view',
  'create',
  'insert',
  'move_lines',
  'get_callable'],
 'prj.vic': ['greet'],
 'safepyrun.core': ['pyrun'],
 'dialoghelper.core': ['msg_insert_line',
  'msg_str_replace',
  'msg_strs_replace',
  'msg_replace_lines',
  'msg_del_lines',
  'msg_pyrun',
  'msg_ast_replace',
  'add_styles',
  'curr_dialog',
  'msg_idx',
  'add_html_a',
  'add_html',
  'iife_a',
  'iife',
  'add_mod',
  'add_mod_a',
  'display_response',
  'read_msg',
  'find_msgs',
  'view_dlg',
  'add_msg',
  'read_msgid',
  'view_msg',
  'del_msg',
  'run_and_prompt',
  'update_msg',
  'run_msg',
  'copy_msg',
  'paste_msg',
  'toggle_header',
  'toggle_bookmark',
  'toggle_comment',
  'url2note',
  'create_or_run_dialog',
  'stop_dialog',
  'load_dialog',
  'rm_dialog',
  'run_code_interactive',
  'ast_py',
  'ast_grep',
  'load_gist',
  'gist_file',
  'import_gist',
  'update_

In [ ]:
get_tool_names('add_msg,run_msg')

{'dialoghelper.core': ['add_msg', 'run_msg']}

In [ ]:
get_tool_names(dialoghelper.tmux)

{'builtins': ['WrapperDescriptorType',
  'MethodWrapperType',
  'MethodDescriptorType',
  'BuiltinFunctionType'],
 'fastcore.imports': ['ipython_shell',
  'in_ipython',
  'in_colab',
  'in_jupyter',
  'in_notebook',
  'is_usable_tool'],
 'fastcore.basics': ['NullType',
  'num_cpus',
  'str2float',
  'str2list',
  'str2date'],
 'fastcore.xtras': ['walk',
  'globtastic',
  'pglob',
  'loads_multi',
  'parse_env',
  'frontmatter',
  'clean_cli_output',
  'rtoken_hex',
  'modify_exception',
  'stringfmt_names',
  'utc2local',
  'local2utc',
  'console_help',
  'type2str',
  'is_typeddict',
  'reawaitable'],
 'toolslm.funccall': ['python'],
 'dialoghelper.tmux': ['shell_ret',
  'set_default_history',
  'pane',
  'list_panes',
  'panes',
  'list_windows',
  'windows',
  'list_sessions',
  'sessions']}

In [ ]:
#| export
delegates(get_tool_names)
def show_tool_names(*args, **kwargs):
    for mn,syms in get_tool_names(*args, **kwargs).items():
        print(mn)
        print('  ', ', '.join(syms))

In [ ]:
show_tool_names()

ipykernel_helper.core
   read_gh_repo, read_url
fastcore.tools
   run_cmd, ensure, valid_path, rg, sed, view, create, insert, move_lines, get_callable
prj.vic
   greet
safepyrun.core
   pyrun
dialoghelper.core
   msg_insert_line, msg_str_replace, msg_strs_replace, msg_replace_lines, msg_del_lines, msg_pyrun, msg_ast_replace, add_styles, curr_dialog, msg_idx, add_html_a, add_html, iife_a, iife, add_mod, add_mod_a, display_response, read_msg, find_msgs, view_dlg, add_msg, read_msgid, view_msg, del_msg, run_and_prompt, update_msg, run_msg, copy_msg, paste_msg, toggle_header, toggle_bookmark, toggle_comment, url2note, create_or_run_dialog, stop_dialog, load_dialog, rm_dialog, run_code_interactive, ast_py, ast_grep, load_gist, gist_file, import_gist, update_gist, read_pr, dialoghelper_explain_dialog_editing, solveit_docs, dialog_link, spawn_agent
pyskills.edit
   str_replace, strs_replace, replace_lines, file_insert_line, file_str_replace, file_strs_replace, file_replace_lines, file_del_lin

In [ ]:
show_tool_names(dialoghelper.stdtools)

dialoghelper.core
   msg_insert_line, msg_str_replace, msg_strs_replace, msg_replace_lines, msg_del_lines, msg_pyrun, msg_ast_replace, add_styles, curr_dialog, msg_idx, add_html_a, add_html, iife_a, iife, add_mod, add_mod_a, display_response, read_msg, find_msgs, view_dlg, add_msg, read_msgid, view_msg, del_msg, run_and_prompt, update_msg, run_msg, copy_msg, paste_msg, toggle_header, toggle_bookmark, toggle_comment, url2note, create_or_run_dialog, stop_dialog, load_dialog, rm_dialog, run_code_interactive, ast_py, ast_grep, load_gist, gist_file, import_gist, update_gist, read_pr, dialoghelper_explain_dialog_editing, solveit_docs, dialog_link, spawn_agent
fastcore.tools
   ensure, valid_path, run_cmd, rg, sed, view, create, insert, move_lines, get_callable
pyskills.edit
   str_replace, strs_replace, replace_lines, file_insert_line, file_str_replace, file_strs_replace, file_replace_lines, file_del_lines, cell_insert_line, cell_str_replace, cell_strs_replace, cell_replace_lines, cell_del_l

In [ ]:
show_tool_names(dialoghelper.core, only_exported=True)

dialoghelper.core
   add_styles, curr_dialog, msg_idx, add_html_a, add_html, iife_a, iife, add_mod, add_mod_a, display_response, read_msg, find_msgs, view_dlg, add_msg, read_msgid, view_msg, del_msg, run_and_prompt, update_msg, run_msg, copy_msg, paste_msg, toggle_header, toggle_bookmark, toggle_comment, url2note, create_or_run_dialog, stop_dialog, load_dialog, rm_dialog, run_code_interactive, msg_insert_line, msg_str_replace, msg_strs_replace, msg_replace_lines, msg_del_lines, msg_pyrun, ast_py, ast_grep, msg_ast_replace, load_gist, gist_file, import_gist, update_gist, read_pr, dialoghelper_explain_dialog_editing, solveit_docs, dialog_link, spawn_agent


In [ ]:
show_tool_names(dialoghelper.tmux)

builtins
   WrapperDescriptorType, MethodWrapperType, MethodDescriptorType, BuiltinFunctionType
fastcore.imports
   ipython_shell, in_ipython, in_colab, in_jupyter, in_notebook, is_usable_tool
fastcore.basics
   NullType, num_cpus, str2float, str2list, str2date
fastcore.xtras
   walk, globtastic, pglob, loads_multi, parse_env, frontmatter, clean_cli_output, rtoken_hex, modify_exception, stringfmt_names, utc2local, local2utc, console_help, type2str, is_typeddict, reawaitable
toolslm.funccall
   python
dialoghelper.tmux
   shell_ret, set_default_history, pane, list_panes, panes, list_windows, windows, list_sessions, sessions


In [ ]:
#| export
def mk_ns_toollist(ns, nms):
    ns = _get_ns(ns)
    return "\n".join(f"- &`{nm}`: {resolve_nm(nm, ns).__doc__}" for nm in nms)

In [ ]:
print(mk_ns_toollist(dialoghelper.tmux, 'shell_ret,pane,list_panes,panes'.split(',')))

- &`shell_ret`: Run shell command locally or over ssh (use host for alias, or ip/user/keyfile)
- &`pane`: Grab the tmux history in plain text
- &`list_panes`: List panes for a session/window (or current if none specified)
- &`panes`: Grab history from all panes in a session/window


In [ ]:
print(mk_ns_toollist(None, ['rg', 'repo2ctx']))

- &`rg`: Run the `rg` command with the args in `argstr`
- &`repo2ctx`: Convert GitHub repo to XML context without cloning


In [ ]:
#| export
delegates(get_tool_names)
async def add_tools_card(ns:Mapping|str=None, **kwargs):
    "Add a message with all tools in namespace `ns` or caller globals"
    ns = _get_ns(ns)
    mod2tool = get_tool_names(ns, **kwargs)
    content = '<!-- tool card -->\n\n' + '\n\n'.join(f"## {mod}\n\n{mk_ns_toollist(ns, tools)}" for mod,tools in mod2tool.items())
    await link_msg(content)

In [ ]:
await add_tools_card()

<!-- tool card -->

## ipykernel_helper.core

- &`read_gh_repo`: Read GitHub repo info: description, file list, and README
- &`read_url`: Read url from web

## fastcore.tools

- &`run_cmd`: Run `cmd` passing split `argstr`, optionally checking for allowed argstr
- &`ensure`: Works like assert b, msg but raise ValueError and is not disabled when run with python -O
- &`valid_path`: Return expanded/resolved Path, raising FileNotFoundError if must_exist and missing
- &`rg`: Run the `rg` command with the args in `argstr`
- &`sed`: Run the `sed` command with the args in `argstr` (e.g for reading a section of a file)
- &`view`: View directory or file contents with optional line range and numbers
- &`create`: Creates a new file with the given content at the specified path
- &`insert`: Insert new_str at specified line number
- &`move_lines`: Move lines from start_line:end_line to before dest_line
- &`get_callable`: Return callable objects defined in caller's module

## prj.vic

- &`greet`: Add a note with `wave` message, and code message with setup python path to the current dialog

## safepyrun.core

- &`pyrun`: Execute restricted Python with access to LLM tools, returning last expression. 
            Runs under asyncio event loop, so `await x` works. 
            `import` works in the usual way. All non-callable globals and non-callable attrs are usable.
            Callable globals are usable only if explicitly registered as tools.
            Callable object attrs are only accessible if `ClassName.method` is registered as a tool.
            Multiline code blocks can be used, including defining functions and variables, for use within the call.
            In addition most builtins are available, plus these symbols: DictReader, HTML, Image, Markdown, Pretty, SVG, add, add_msg, ast_grep, cell_del_lines, cell_insert_line, cell_replace_lines, cell_str_replace, cell_strs_replace, chmod, close, contextmanager, copy, copy2, copy_msg, copytree, create, create_or_run_dialog, curr_dialog, del_msg, dialog_link, dialoghelper_explain_dialog_editing, display, doc, docfind, dumps, exists, expanduser, file_del_lines, file_insert_line, file_replace_lines, file_str_replace, file_strs_replace, find, find_msgs, findall, flush, fromstring, gather, get, getsizeof, glob, hardlink_to, importmodule, insert, is_dir, is_file, iter, iterdir, joinpath, list_panes, list_pyskills, list_sessions, list_windows, load_dialog, loads, match, md, mkdir, move, move_lines, msg_ast_replace, msg_del_lines, msg_idx, msg_insert_line, msg_pyrun, msg_replace_lines, msg_str_replace, msg_strs_replace, open, options, pane, panes, paste_msg, pyrun, quote, read, read_bytes, read_gh_repo, read_msg, read_msgid, read_pr, read_text, read_url, reader, readline, readlines, relative_to, rename, replace, replace_lines, resolve, rg, rmdir, rmtree, run_code_interactive, save, sed, seek, sessions, sleep, solveit_docs, spawn_agent, stat, stop_dialog, str_replace, strs_replace, suppress, symdir, symfiles_folder, symfiles_package, symlen, symlink_to, symnth, symsearch, symslice, symsrc, symtype, symval, tell, toggle_bookmark, toggle_comment, toggle_header, tostring, touch, unlink, unquote, update_msg, urlencode, view, view_cell, view_dlg, view_msg, view_nb, windows, with_name, with_suffix, write, write_bytes, write_text, writelines, xdir

            Allowed methods by type: `BufferedRandom`: close, flush, read, readline, readlines, seek, tell, write, writelines; `BufferedWriter`: close, flush, read, readline, readlines, seek, tell, write, writelines; `BytesIO`: *; `Counter`: *; `Element`: find, findall, get, iter; `FileIO`: close, flush, read, readline, readlines, seek, tell, write, writelines; `IPython.core.display`: HTML, Image, Markdown, Pretty, SVG; `IPython.core.display_functions`: display; `Notebook`: add, find, md, move, open, save, view; `Path`: chmod, exists, expanduser, glob, hardlink_to, is_dir, is_file, iterdir, joinpath, match, mkdir, read_bytes, read_text, relative_to, rename, replace, resolve, rmdir, stat, symlink_to, touch, unlink, with_name, with_suffix, write_bytes, write_text; `Pattern`: *; `StringIO`: *; `TextIOWrapper`: close, flush, read, readline, readlines, seek, tell, write, writelines; `ast`: *; `asyncio`: gather, sleep; `base64`: *; `binascii`: *; `bisect`: *; `bytes`: *; `cmath`: *; `collections`: *; `colorsys`: *; `contextlib`: contextmanager, suppress; `copy`: *; `csv`: DictReader, reader; `dataclasses`: *; `datetime`: *; `datetime`: *; `decimal`: *; `deque`: *; `dict`: *; `difflib`: *; `enum`: *; `float`: *; `fnmatch`: *; `fractions`: *; `frozenset`: *; `functools`: *; `hashlib`: *; `heapq`: *; `html`: *; `httpx`: get, options; `inspect`: *; `int`: *; `ipaddress`: *; `itertools`: *; `json`: *; `keyword`: *; `list`: *; `math`: *; `operator`: *; `pickle`: dumps, loads; `posixpath`: *; `pprint`: *; `pyskills.core`: doc, docfind, xdir; `pyskills.edit`: cell_del_lines, cell_insert_line, cell_replace_lines, cell_str_replace, cell_strs_replace, file_del_lines, file_insert_line, file_replace_lines, file_str_replace, file_strs_replace, view_cell, view_nb; `random`: *; `re`: *; `secrets`: *; `set`: *; `shlex`: *; `shutil`: copy, copy2, copytree, move, rmtree; `statistics`: *; `str`: *; `struct`: *; `sys`: getsizeof; `textwrap`: *; `time`: *; `traceback`: *; `tuple`: *; `types`: *; `unicodedata`: *; `urllib.parse`: *; `uuid`: *; `warnings`: *; `xml.etree.ElementTree`: fromstring, tostring; `zlib`: *

            **NB**: Locals are exported back to the caller's namespace unless they'd shadow an existing callable or module.
            - Symbols ending with `_` are always exported, even if they shadow existing names.
            Examples: `len([1,2,3])` (builtin); `add_msg(content="hi")` (tool); `df.shape` (non-callable attr);
            `[x**2 for x in range(5)]` (last expression returned); `sorted(my_dict.items())` (builtin + non-callable attr)

## dialoghelper.core

- &`msg_insert_line`: Insert new_str at specified line number
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_str_replace`: Replace occurrence(s) of old_str with new_str
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_strs_replace`: Replace multiple strings simultaneously
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_replace_lines`: Replace line range with new content
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_del_lines`: Delete line range
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_pyrun`: Edit text by running `code` in pyrun. `text` var has content, last expr is new content
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`msg_ast_replace`: Replace code by AST pattern using ast-grep.

    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence

    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)
Be sure you've called `view_msg(…)` to ensure you know the line nums.
If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
update_output: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`add_styles`: Add solveit styles to `s`
- &`curr_dialog`: Get the current dialog info.
- &`msg_idx`: Get absolute index of message in dialog.
- &`add_html_a`: Send HTML to the browser to be swapped into the DOM
- &`add_html`: Send HTML to the browser to be swapped into the DOM
- &`iife_a`: Wrap javascript code string in an IIFE and execute it via `add_html`
- &`iife`: Wrap javascript code string in an IIFE and execute it via `add_html`
- &`add_mod`: Wrap javascript code string in a js script module and add it via `add_html`
- &`add_mod_a`: Wrap javascript code string in a js script module and add it via `add_html`
- &`display_response`: Return a special response where `display` is added as markdown/HTML to the prompt output, and `result` is returned to the LLM
- &`read_msg`: Get the message indexed in the current dialog.
    NB: Messages in the current dialog above the current message are *already* visible; use this only when you need line numbers for editing operations, or for messages not in the current dialog or below the current message.
    - To get the exact message use `n=0` and `relative=True` together with `id`.
    - To get a relative message use `n` (relative position index).
    - To get the nth message use `n` with `relative=False`, e.g `n=0` first message, `n=-1` last message.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
- &`find_msgs`: Often it is more efficient to call `view_dlg` to see the whole dialog at once, so you can use it all from then on, instead of using `find_msgs`.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
    Message ids are identical to those in LLM chat history, so do NOT call this to view a specific message if it's in the chat history--instead use `view_msg`.
    Do NOT use find_msgs to view message content in the current dialog above the current prompt -- these are *already* provided in LLM context, so just read the content there directly. (NB: LLM context only includes messages *above* the current prompt, whereas `find_msgs` can access *all* messages.)
    To refer to a found message from code or tools, use its `id` field.
- &`view_dlg`: Concise XML view of all messages (optionally filtered by type), not including metadata. Often it is more efficient to call this to see the whole dialog at once (including line numbers if needed), instead of running `find_msgs` or `view_msg` multiple times.
- &`add_msg`: Add/update a message to the queue to show after code execution completes, and optionally run it.
    Code messages are run using pyrun's restricted sandbox.
    **NB**: when creating multiple messages in a row, after the 1st message set `id` to the result of the last `add_msg` call,
    otherwise messages will appear in the dialog in REVERSE order.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
- &`read_msgid`: Get message `id`. Message IDs can be view directly in LLM chat history/context, or found in `find_msgs` results.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
- &`view_msg`: Views the *content* of message `id`. Same as `read_msgid(...)['content']`, defaulting to `nums=True`.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
- &`del_msg`: Delete a message from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages.
- &`run_and_prompt`: Run `code` and then run `prompt`, returning the resulting message ID.
- &`update_msg`: Update an existing message. Provide either `msg` OR field key/values to update.
    - Use `content` param to update contents.
    - Only include parameters to update--missing ones will be left unchanged.
    If `dname` is None, the current dialog is used. If it is an open dialog, it will be updated interactively with real-time updates to the browser. If it is a closed dialog, it will be updated on disk. Dialog names must be paths relative to solveit root (if starting with `/`, e.g. `/myproject/dlg`) or relative to the current dialog's folder (if not starting with `/`), and should *not* include the .ipynb extension. **Use absolute paths when targeting dialogs outside the current dialog's folder tree.**
- &`run_msg`: Adds a message to the run queue. Use read_msg to see the output once it runs.
- &`copy_msg`: Add `ids` to clipboard.
- &`paste_msg`: Paste clipboard msg(s) after/before the current selected msg (id).
- &`toggle_header`: Toggle collapsed header state for `id`
- &`toggle_bookmark`: Toggle numbered bookmark (1-9) on a message, clearing it from any other message when setting
- &`toggle_comment`: Toggle line comments on code message(s). If any lines are uncommented, comments all; otherwise uncomments all.
- &`url2note`: Read URL as markdown, and add note(s) below current message with the result
- &`create_or_run_dialog`: Create a new dialog, or set an existing one running
- &`stop_dialog`: Stop a running dialog kernel
- &`load_dialog`: Run all code messages from `src_dname` into the target dialog's kernel and return dialog contents.
- &`rm_dialog`: Delete a dialog (or folder) and associated records, stopping the kernel if running
- &`run_code_interactive`: Insert code into user's dialog and request for the user to run it. Use other tools where possible, 
    but if they can not find needed information, *ALWAYS* use this instead of guessing or giving up.
    IMPORTANT: This tool is TERMINAL - after calling it, you MUST stop all tool usage 
    and wait for user response. Never call additional tools after this one.
- &`ast_py`: Get an SgRoot root node for python `code`
- &`ast_grep`: Use `ast-grep` to find code patterns by AST structure (not text).
    
    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence
    
    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)
    
    Useful for: Refactoring—find all uses of deprecated APIs or changed signatures; Security review—locate SQL queries, file operations, eval calls; Code exploration—understand how libraries are used across codebase; Pattern analysis—find async functions, error handlers, decorators; Better than regex—handles multi-line code, nested structures, respects syntax
- &`load_gist`: Retrieve a gist
- &`gist_file`: Get the first file from a gist
- &`import_gist`: Import gist directly from string without saving to disk
- &`update_gist`: Update the first file in a gist with new content
- &`read_pr`: Fetch a GitHub PR or issue's title, body, optionally replies, and diff (if PR)
- &`dialoghelper_explain_dialog_editing`: Call this to get a detailed explanation of how dialog editing is done in dialoghelper.
    **ALWAYS** call this first, if dialog editing has not previously occured in this session
- &`solveit_docs`: Full reference documentation for Solveit - use this to answer questions about how to use Solveit.
    **NB**: The whole docs fit in LLM context, so read the whole thing, don't search/filter it. *Always* re-run rather than relying on truncated history or assumptions.
- &`dialog_link`: Return an IPython HTML link to open a dialog in Solveit.
    After calling this tool, output the resulting HTML anchor tag exactly as returned—do not wrap in a fenced code block or convert to markdown link format.
- &`spawn_agent`: Spawn a subagent to complete a task defined by `prompt`. Must be run as a tool - not from Python.
    The subagent's context and tools is defined by the parent prompt's history

## pyskills.edit

- &`str_replace`: Replace occurrence(s) of old_str with new_str
- &`strs_replace`: Replace multiple strings simultaneously
- &`replace_lines`: Replace line range with new content
- &`file_insert_line`: Insert new_str at specified line number
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_str_replace`: Replace occurrence(s) of old_str with new_str
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_strs_replace`: Replace multiple strings simultaneously
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_replace_lines`: Replace line range with new content
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_del_lines`: Delete line range
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`cell_insert_line`: Insert new_str at specified line number
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`cell_str_replace`: Replace occurrence(s) of old_str with new_str
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`cell_strs_replace`: Replace multiple strings simultaneously
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`cell_replace_lines`: Replace line range with new content
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`cell_del_lines`: Delete line range
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages
- &`insert_line`: Insert new_str at specified line number
- &`del_lines`: Delete line range
- &`view_cell`: Show cell source with optional line numbers
- &`view_nb`: Show notebook source as concise xml, optionally including output if `incl_out`

## toolslm.xml

- &`json_to_xml`: Convert `d` to XML.
- &`mk_doctype`: Create a `doctype` named tuple
- &`docs_xml`: Create an XML string containing `docs` in Anthropic's recommended format
- &`files2ctx`: Convert files to XML context, handling notebooks
- &`folder2ctx`: Convert folder contents to XML context, handling notebooks
- &`folder2ctx_cli`: CLI to convert folder contents to XML context, handling notebooks
- &`repo2ctx`: Convert GitHub repo to XML context without cloning
- &`repo2ctx_cli`: CLI to convert GitHub repo contents to XML context

## toolslm.inspecttools

- &`importmodule`: Import a module into the caller's global namespace so it's available for `symsrc`, `symval`, `symdir`, etc.
    Use this before inspecting or using symbols from modules not yet imported.
- &`resolve`: Resolve a dotted symbol string to its Python object, with optional [n] indexing.
    Sets global `_last` to the resolved object for chaining.
    Pass `"_last"` to reference the result of the previous tool call.

    Examples:

    - `resolve("sympy.sets.sets.Interval")` -> `<class 'sympy.sets.sets.Interval'>`
    - `resolve("mylist[2]")` -> third element of mylist
- &`symsrc`: Get the source code for a symbol.

    Examples:

    - `symsrc("Interval")` -> source code of Interval class if it's already imported
    - `symsrc("sympy.sets.sets.Interval")` -> source code of Interval class
    - `symsrc("_last")` -> source of object from previous tool call
    - For dispatchers or registries of callables: `symnth("module.dispatcher.funcs", n) then symsrc("_last")`
- &`symtype`: Get the type of a symbol and set `_last`.

    Examples:

    - `symtype("sympy.sets.sets.Interval")` -> `<class 'type'>`
    - `symtype("doesnotexist")` -> `'SymbolNotFound`
    - `symtype("_last")` -> type of previous result
- &`symval`: List of repr of symbols' values.

    Examples:
    
    - `symval("sympy.sets.sets.Interval")` -> `[<class 'sympy.sets.sets.Interval'>]`
    - `symval("some_dict.keys")` -> `[dict_keys([...])]`
    - `symval("a,notexist")` -> `['foo','SymbolNotFound']`
- &`symtype_val`: List of 2-ple of (type,repr) of symbols' values.

    Examples:
    
    - `symtype_val("a,c,notexist")` -> `[(<class 'str'>,'foo'),(<class 'int'>,1), 'SymbolNotFound']`
- &`symdir`: Get dir() listing of a symbol's attributes and set `_last`. E.g: `symdir("sympy.Interval")` -> `['__add__', '__and__', ...]`
- &`symnth`: Get the nth value from a dict (or any object with .values()). Sets `_last` so you can chain with `symsrc("_last")` etc.

    Examples:
    
    - `symnth("dispatcher.funcs", 12)` -> 13th registered function
    - `symnth("dispatcher.funcs", 0); symsrc("_last")` -> source of first handler
- &`symlen`: Returns the length of the given symbol
- &`symslice`: Returns the contents of the symbol from the given start to the end.
- &`symsearch`: Search contents of symbol, which is assumed to be str for regex, or iterable for non-regex.
    Regex mode returns (match, start, end) tuples; otherwise returns (item, index) tuples
- &`symset`: Set _ai_sym to the given value
- &`symfiles_folder`: Return XML context of files in the folder containing `sym`'s definition
- &`symfiles_package`: Return XML context of all files in `sym`'s top-level package

## pyskills.core

- &`list_pyskills`: Returns {module: description} for all pyskills. To load a module, use `import {module}` then view `doc({module}).
    **NB**: pyskills are *THE* critical way to extend functionality. *ALWAYS* check for pyskills to help with tasks.
    If unsure whether a particular pyskill might help, load it and grabs its docs to see!

## __main__

- &`hydrate`: Traverse dialog looking for linked messages to update `__linked_msgs`
- &`solveit_version`: Return the version of solveit if it is found
- &`in_dialog`: Check if the code is running in a solveit dialog
- &`get_caller_globals`: Return the globals of the caller
- &`find_var`: Search for var in all frames of the call stack
- &`has_tag`: Check if string contains tags created by get_tag at the start or end of the string
- &`find_tag`: Find tag in a string and return their details
- &`next_filename`: Get next available duplicate filename for path, e.g., 'img.jpg' -> 'img_dup1.jpg'
- &`info`: Returns information about the dialog
- &`add_info`: Add a message with information about the dialog
- &`get_tool_names`: Return dict mapping module names to lists of usable tool names from namespace ns (or IPython user namespace if None).

## inspect

- &`currentframe`: Return the frame of the caller or None if this is not possible.

## anyio

- &`sleep`: 
    Pause the current task for the specified duration.

    :param delay: the duration, in seconds

    

## anyio.from_thread

- &`start_blocking_portal`: 
    Start a new event loop in a new thread and run a blocking portal in its main task.

    The parameters are the same as for :func:`~anyio.run`.

    :param backend: name of the backend
    :param backend_options: backend options
    :param name: name of the thread
    :return: a context manager that yields a blocking portal

    .. versionchanged:: 3.0
        Usage as a context manager is now required.

    

## fastcore.imports

- &`in_ipython`: Check if code is running in some kind of IPython environment
- &`is_usable_tool`: True if the function has a docstring and all parameters have types, meaning that it can be used as an LLM tool.
<!-- linkedto: _9bd38272 -->

In [ ]:
await add_tools_card(dialoghelper.tmux, only_exported=True)

<!-- tool card -->

## dialoghelper.tmux

- &`shell_ret`: Run shell command locally or over ssh (use host for alias, or ip/user/keyfile)
- &`set_default_history`: Set the default number of lines to capture from tmux history
- &`pane`: Grab the tmux history in plain text
- &`list_panes`: List panes for a session/window (or current if none specified)
- &`panes`: Grab history from all panes in a session/window
- &`list_windows`: List all windows in a session
- &`windows`: Grab history from all panes in all windows of a session
- &`list_sessions`: List all tmux sessions
- &`sessions`: Grab history from all panes in all windows of all sessions
<!-- linkedto: _11232531 -->

In [ ]:
await add_tools_card('rg, ensure')

<!-- tool card -->

## fastcore.tools

- &`rg`: Run the `rg` command with the args in `argstr`
- &`ensure`: Works like assert b, msg but raise ValueError and is not disabled when run with python -O
<!-- linkedto: _c877d4ad -->

## export -

In [ ]:
from pote.dutil import ctxusage
await ctxusage()

22731

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

###### start-section

In [ ]:
from dutil.gitutil import Git

In [ ]:
g = Git('')
g.cstatus()

In [ ]:
g.commit('-m', 'renamed to 00_git')

In [ ]:
g.add('07_git.ipynb')

In [ ]:
g.cdiff('../dutil/logger.py')

###### end-section

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()